![Itq](img/LogoITQ.jpg)

**Propueta para la implementación de Fast Api para uso de neurona MP**

![Python](img/python_logo.png)

* *Nombre:* Judith Piedra
* *Fecha :* **13-10-2026**
* *Repositorio:* <a href= "https://github.com/MerakiDevJP/machine2_23092026nocturno.git"> Link del repositorio Github</a>
* https://github.com/MerakiDevJP/machine2_23092026nocturno.git

# Plan de desarrollo Fast Api

**Fase 1**
**Análisis y Selección de Características (Feature Selection)**

El objetivo de esta fase es determinar las 5 características de mayor poder diagnóstico para utilizar en la Fast Api.

1. Importar el dataset de Cáncer de Mama de Wisconsin desde scikit-learn y estructurarlo en un DataFrame de Pandas.

2. Aplicar un modelo auxiliar (como Random Forest o un análisis de correlación de Pearson) para identificar qué variables aportan la mayor ganancia de información.

3. Seleccionar formalmente las 5 características críticas basadas en patología tumoral y dispersión estadística, y saber si coinciden con las variables investigadas: worst radius, worst perimeter, worst concave points, worst area, mean concave points

Justificación de selección:
- La etiqueta worst representan el promedio de los tres valores más altos encontrados en las células analizadas de la muestra. En un tumor maligno, siempre existen áreas de crecimiento agresivo o "nódulos satélites" donde las dimensiones celulares alcanzan sus valores más extremos. Estas variaciones extremas son el indicador más claro de malignidad.
- worst radius (Radio peor) y worst area (Área peor), los tumores malignos ocupan mucho más espacio y crecen con mayor volumen. El radio y el área máxima reflejan directamente la expansión tridimensional de la lesión en el tejido mamario.
- worst perimeter (Perímetro peor), el contorno de una célula o grupo celular maligno suele ser rugoso, anguloso y deforme debido a la rápida división celular. Un perímetro máximo elevado combinado con un área grande delata una estructura celular altamente desorganizada.
- worst concave points y mean concave points (Puntos cóncavos), los "puntos cóncavos" miden las indentaciones o porciones hacia adentro en el contorno de la forma nuclear o celular. Las células sanas o benignas tienden a tener formas ovaladas o circulares regulares (convexas). Por el contrario, los núcleos de las células cancerosas presentan pliegues, muescas y deformaciones profundas (múltiples puntos cóncavos). Estadísticamente, en la literatura médica y de minería de datos de este dataset, los concave points son históricamente las variables con mayor correlación lineal directa frente al diagnóstico de malignidad.



**Fase 2**
**Preprocesamiento y Entrenamiento de la Neurona MP**

Neurona procesada en el documento 11.CasoPractivoNeuronaMP

**Fase 3**
**Construcción del Backend (FastAPI)**

Crear la pasarela lógica que mantendrá el modelo entrenado en memoria y procesará las peticiones HTTP del usuario.

- Configuración del entorno: Instalar las librerías necesarias (fastapi, uvicorn, pandas, numpy, scikit-learn).

- Estructura del servidor (main.py):
    * Automatizar el entrenamiento del modelo al iniciar la aplicación para que esté listo para predecir.
    * Definir un esquema de validación con Pydantic que asegure la recepción exacta de las 5 variables requeridas.
      
- Creación de Endpoints:
    * Endpoint de la interfaz (/): Retornará el código HTML del formulario.
    * Endpoint de predicción (/): Recibirá los datos del paciente, los binarizará, ejecutará el método .predict() de la neurona y devolverá el diagnóstico ("Benigno" o "Maligno").



In [4]:
# ==========================================
# 1. CARGA Y EXPLORACIÓN INICIAL
# ==========================================
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# Cargar el dataset de Cáncer de Mama de Wisconsin
cancer_mama = load_breast_cancer()

# Estructurarlo en un DataFrame de Pandas con los nombres reales de las características
df_full = pd.DataFrame(cancer_mama.data, columns=cancer_mama.feature_names)
Y = cancer_mama.target

print(f"Dimensiones del dataset completo: {df_full.shape}")
print("Primeras características del dataset:")
display(df_full.head(3))




Dimensiones del dataset completo: (569, 30)
Primeras características del dataset:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [5]:
# ==========================================
# 2. EVALUACIÓN DE IMPORTANCIA (RANDOM FOREST)
# ==========================================
# Entrenamos un modelo de Bosques Aleatorios para medir la ganancia de información (Feature Importance)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(df_full, Y)

# Crear un DataFrame con la importancia de cada variable
feature_importances = pd.DataFrame({
    'Característica': df_full.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values(by='Importancia', ascending=False)

print("\n--- Ranking de Importancia de Variables (Top 10) ---")
display(feature_importances.head(10))




--- Ranking de Importancia de Variables (Top 10) ---


,Característica,Importancia
23,worst area,0.139357
27,worst concave points,0.132225
7,mean concave points,0.107046
20,worst radius,0.082848
22,worst perimeter,0.080850
2,mean perimeter,0.067990
6,mean concavity,0.066917
3,mean area,0.060462
26,worst concavity,0.037339
0,mean radius,0.034843


In [6]:

# ==========================================
# 3. FILTRADO DE LAS 5 VARIABLES CLAVE
# ==========================================
# Se seleccionan las 5 características críticas basadas en patología tumoral y dispersión estadística
top_5_features = [
    'worst area',
    'worst concave points',
    'mean concave points',
    'worst radius', 
    'worst perimeter'    
]
# DataFrame reducido para analizar si luego se usa en la Neurona MP
df_reducido = df_full[top_5_features]

print("\n--- DataFrame Reducido con las 5 Características Clave ---")
display(df_reducido.head(5))


--- DataFrame Reducido con las 5 Características Clave ---


,worst area,worst concave points,mean concave points,worst radius,worst perimeter
0,2019.0,0.2654,0.14710,25.38,184.60
1,1956.0,0.1860,0.07017,24.99,158.80
2,1709.0,0.2430,0.12790,23.57,152.50
3,567.7,0.2575,0.10520,14.91,98.87
4,1575.0,0.1625,0.10430,22.54,152.20


**Fase 4**
**Desarrollo de la Interfaz de Usuario (Frontend HTML/Bootstrap)**

Construir una interfaz limpia, ligera y fácil de usar que evite la fricción de ingresar 30 parámetros clínicos.
- Diseño del Formulario: Crear un panel central utilizando Bootstrap con 5 campos de entrada numéricos (input type="number"), uno para cada característica seleccionada.
- Lógica de Conexión (JavaScript):
    * Capturar el evento de envío (submit) del formulario.
    * Empaquetar los valores en formato JSON.
    * Enviar una petición asíncrona mediante fetch() hacia la ruta /predict de la API de FastAPI.
- Visualización de Resultados: Mostrar dinámicamente una alerta visual en pantalla indicando el resultado del diagnóstico de forma inmediata.



**Fase 5**
**Pruebas, Validación y Despliegue**

Garantizar el correcto funcionamiento del sistema completo.
